In [10]:
# Importer les bibliothéques nécessaires

# Pour le calcul des distances euclidiennes
from scipy.spatial import distance as dist
# Pour l'extraction et manipulation des caractéristiques faciaux
from imutils import face_utils
# Pour travailler avec les Numpy arrays
import numpy as np
# Pour changer le dimension de trames capturées
import imutils
# Pour la détection du visage
import dlib
# Pour la capture de vidéo
import cv2
# Pour utiliser dans les conditions
import time


In [11]:
def rapport_aspect_oeil(oeil):
    """
    Calcule le rapport d'aspect d'oeil
    Arguments :
        oeil (dtype = dictionnaire) : Représente l'oeil gauche ou droite
    Outputs : 
        ear (dtype = float) : Représente le rapport d'aspect d'oeil
    """
    
    # Calculer la distance euclidienne entre deux points verticales d'indice 1 et 5
    A = dist.euclidean(oeil[1], oeil[5])
    
    # Calculer la distance euclidienne entre deux points verticales d'indice 1 et 5
    B = dist.euclidean(oeil[2], oeil[4])
    
    # Calculer la distance euclidienne entre les deux points horizontales d'indice 0 et 3
    C = dist.euclidean(oeil[0], oeil[3])
    
    # Calculer le rapport d'aspect d'oeil
    ear = (A + B) / (C * 2.0)
    
    return ear

In [12]:
def rapport_aspect_bouche(bouche):
    """
    Calcule le rapport d'aspect de la bouche
    Arguments : 
        bouche (dtype = dictionnaire) : Représente la bouche
    Outputs:
        mar (dtype = float) : Représente l'aspect de la bouche
    """
    # Calculer la distance euclidienne entre deux points verticales  d'indice 1 et 5 
    Y1 = dist.euclidean(bouche[1], bouche[7])
    
    # Calculer la distance euclidienne entre deux points verticales d'indice 2 et 4
    Y2 = dist.euclidean(bouche[2], bouche[6])

    # Calculer la distance euclidienne entre deux points verticales d'indice 3 et 5
    Y3 = dist.euclidean(bouche[3], bouche[5])
    
    # Calculer la distance euclidienne entre les deux points horizontales d'indice 0 et 4
    X = dist.euclidean(bouche[0], bouche[4])
    
    # Calculer le rapport d'aspect d'oeil
    mar = (Y1 + Y2 + Y3) / (X * 2.0)
    
    return mar

In [13]:
# Définition des différents seuils

SEUIL_ASPECT_YEUX = 0.2
NB_TRAMES_SEUIL_ASPECT_YEUX = 45
SEUIL_ASPECT_BOUCHE = 0.5
TAUX_CARDIAQUE_NORMAL = 80
AGE = 0
sexe = "MALE"

In [14]:
# Compter le nombre des trames que les yeux se clignes
counter = 0
# Condition pour additionner le counter
counter_condition = False
# Condition pour vérifier le baillement
état_baillement = False
# Calculer la durée du baillement
yawning_time = 0

trigger_time_set = False

trigger_time_set_clignement = False
# Nombre de clignement
clignement = 0
# Condition pour vérifier le clignement
etat_clignement = False
# Nombre de baillements
nb_baillements = 0
# Instant du 1er baillement
temps_baillement = 0
# Instant du 1er clignement
temps_clignement = 0
# Etat du conducteur
état_conducteur = "normal"

In [15]:
# Définition du détection du visage par dlib
détecteur = dlib.get_frontal_face_detector()

# Donner le chemin au prédicteur du visage 
chemin_prédicteur = "C:/Users/MSI/Downloads/shape_predictor_68_face_landmarks.dat"

# Création du points de caractéristiques faciaux
prédicteur = dlib.shape_predictor(chemin_prédicteur)

In [16]:
# Prendre les indices de l'oeil droite
(d_début, d_fin) = face_utils.FACIAL_LANDMARKS_IDXS["right_eye"]

# Prendre les indices de l'oeil gauche
(g_début, g_fin) = face_utils.FACIAL_LANDMARKS_IDXS["left_eye"]

# Prendre les indices de la bouche
(b_début, b_fin) = face_utils.FACIAL_LANDMARKS_IDXS["inner_mouth"]

In [17]:
# Déclaration du webcam 
caméra = cv2.VideoCapture(0) 

In [18]:
# Initialation de capture du video
while True:
    """
    Prend la trame capturée par la caméra,
    change son largeur en 640
    et la convertit en greyscale channels
    pour simplifier le processing de la trame.
    """
    ret, trame = caméra.read()
    trame = imutils.resize(trame, width = 640)
    trame_gris = cv2.cvtColor(trame, cv2.COLOR_BGR2GRAY)
    
    # Détecter les visages dans la trame en niveaux de gris
    rectangles = détecteur(trame_gris, 0)
    if len(rectangles) >= 1:
        # Sélectionner le premier visage détectée
        rectangle = rectangles[0]
        if rectangle:
            """
            Prédiction du points de caractéristique faciaux,
            puis stocker leur coordonnées (x, y) 
            dans un NumPy array pour performance plus efficace
            """
            shape = prédicteur(trame_gris, rectangle)
            shape = face_utils.shape_to_np(shape)

            """
            Extraction du coordonnées des points qui représente
            les yeux, utilise ces coordonnées pour calculer le 
            ear général
            """
            oeil_gauche = shape[g_début : g_fin]
            ear_gauche = rapport_aspect_oeil(oeil_gauche)

            oeil_droite = shape[d_début : d_fin]
            ear_droite = rapport_aspect_oeil(oeil_droite)

            ear_général = (ear_droite + ear_gauche) / 2
            ear_formatté = "{:.2f}".format(ear_général)

            bouche = shape[b_début : b_fin]
            mar_bouche = rapport_aspect_bouche(bouche)
            mar_formatté = "{:.2f}".format(mar_bouche)

            """
            Dessiner un polygone qui enveloppe les points
            de la bouche ou de l'oeil droite ou gauche
            """
            conv_oeil_gauche = cv2.convexHull(oeil_gauche)
            conv_oeil_droite = cv2.convexHull(oeil_droite)
            conv_bouche = cv2.convexHull(bouche)

            # Coloriser les polygones
            cv2.drawContours(trame, [conv_oeil_gauche], -1, (0, 255, 0), 1)
            cv2.drawContours(trame, [conv_oeil_droite], -1, (0, 255, 0), 1)
            cv2.drawContours(trame, [conv_bouche], -1, (0, 255, 255), 1)


        taux_cardiaque = 80
        # Détection de la fatigue à partir du taux cardiaque
        if sexe == "FEMALE":
            if 18 <= AGE <= 25:
                    if 61 <= taux_cardiaque <=65:
                        état_conducteur = "Excellente"
                    elif  66 <= taux_cardiaque <=69 :
                        état_conducteur = "Bonne"
                    elif  74 <= taux_cardiaque <=78 : 
                        état_conducteur = "Normale"
                    elif  79 <= taux_cardiaque <=84 : 
                         état_conducteur = "Moins bonne"
                    elif taux_cardiaque <85 : 
                        état_conducteur = "Mauvaise"
            if 26 <= AGE <= 35:
                    if 60 <= taux_cardiaque <=64:
                        état_conducteur = "Excellente"
                    elif  65 <= taux_cardiaque <=68 :
                        état_conducteur = "Bonne"
                    elif  73 <= taux_cardiaque <=76 : 
                        état_conducteur = "Normale"
                    elif  77 <= taux_cardiaque <=82 : 
                         état_conducteur = "Moins bonne"
                    elif taux_cardiaque <83 : 
                        état_conducteur = "Mauvaise"
            if 36 <= AGE <= 45:
                    if 60 <= taux_cardiaque <=64:
                        état_conducteur = "Excellente"
                    elif  65 <= taux_cardiaque <=69 :
                        état_conducteur = "Bonne"
                    elif  74 <= taux_cardiaque <=78 : 
                        état_conducteur = "Normale"
                    elif  79 <= taux_cardiaque <=84 : 
                         état_conducteur = "Moins bonne"
                    elif taux_cardiaque <85 : 
                        état_conducteur = "Mauvaise"
            if 46 <= AGE <= 55:
                    if 61 <= taux_cardiaque <=65:
                        état_conducteur = "Excellente"
                    elif  66 <= taux_cardiaque <=69 :
                        état_conducteur = "Bonne"
                    elif  74 <= taux_cardiaque <=77 : 
                        état_conducteur = "Normale"
                    elif  78 <= taux_cardiaque <=83 : 
                         état_conducteur = "Moins bonne"
                    elif taux_cardiaque <84 : 
                        état_conducteur = "Mauvaise"

            if 56 <= AGE <= 65:
                    if 60 <= taux_cardiaque <=64:
                        état_conducteur = "Excellente"
                    elif  65 <= taux_cardiaque <=68 :
                        état_conducteur = "Bonne"
                    elif  74 <= taux_cardiaque <=77 : 
                        état_conducteur = "Normale"
                    elif  78 <= taux_cardiaque <=83 : 
                         état_conducteur = "Moins bonne"
                    elif taux_cardiaque <85 : 
                        état_conducteur = "Mauvaise"
            if  AGE > 65:
                    if 60 <= taux_cardiaque <=64:
                        état_conducteur = "Excellente"
                    elif  65 <= taux_cardiaque <=68 :
                        état_conducteur = "Bonne"
                    elif  73 <= taux_cardiaque <=76 : 
                        état_conducteur = "Normale"
                    elif  77 <= taux_cardiaque <=84 : 
                         état_conducteur = "Moins bonne"
                    elif taux_cardiaque <84 : 
                        état_conducteur = "Mauvaise"
        elif sexe == "MALE":
            if 18 <= AGE <= 25:
                    if 56 <= taux_cardiaque <=61:
                        état_conducteur = "Excellente"
                    elif  62<= taux_cardiaque <=65 :
                        état_conducteur = "Bonne"
                    elif  70 <= taux_cardiaque <=73 : 
                        état_conducteur = "Normale"
                    elif  74 <= taux_cardiaque <=81 : 
                         état_conducteur = "Moins bonne"
                    elif taux_cardiaque <82 : 
                        état_conducteur = "Mauvaise"
            if 26 <= AGE <= 35:
                    if 55 <= taux_cardiaque <=61:
                        état_conducteur = "Excellente"
                    elif  62 <= taux_cardiaque <=65 :
                        état_conducteur = "Bonne"
                    elif  71 <= taux_cardiaque <=74 : 
                        état_conducteur = "Normale"
                    elif  75 <= taux_cardiaque <=81 : 
                         état_conducteur = "Moins bonne"
                    elif taux_cardiaque <82 : 
                        état_conducteur = "Mauvaise"
            if 36 <= AGE <= 45:
                    if 57 <= taux_cardiaque <=62:
                        état_conducteur = "Excellente"
                    elif  63 <= taux_cardiaque <=66 :
                        état_conducteur = "Bonne"
                    elif  71 <= taux_cardiaque <=75 : 
                        état_conducteur = "Normale"
                    elif  76 <= taux_cardiaque <=82 : 
                         état_conducteur = "Moins bonne"
                    elif taux_cardiaque <83 : 
                        état_conducteur = "Mauvaise"
            if 46 <= AGE <= 55:
                    if 58 <= taux_cardiaque <=63:
                        état_conducteur = "Excellente"
                    elif  64 <= taux_cardiaque <=67 :
                        état_conducteur = "Bonne"
                    elif  72 <= taux_cardiaque <=76 : 
                        état_conducteur = "Normale"
                    elif  77 <= taux_cardiaque <=83 : 
                         état_conducteur = "Moins bonne"
                    elif taux_cardiaque <84 : 
                        état_conducteur = "Mauvaise"

            if 56 <= AGE <= 65:
                    if 57 <= taux_cardiaque <=61:
                        état_conducteur = "Excellente"
                    elif  62 <= taux_cardiaque <=67 :
                        état_conducteur = "Bonne"
                    elif  72 <= taux_cardiaque <=75 : 
                        état_conducteur = "Normale"
                    elif  76 <= taux_cardiaque <=81 : 
                         état_conducteur = "Moins bonne"
                    elif taux_cardiaque <82 : 
                        état_conducteur = "Mauvaise"
            if  AGE > 65:
                    if 56 <= taux_cardiaque <=61:
                        état_conducteur = "Excellente"
                    elif  62 <= taux_cardiaque <=65 :
                        état_conducteur = "Bonne"
                    elif  70 <= taux_cardiaque <=73 : 
                        état_conducteur = "Normale"
                    elif  74 <= taux_cardiaque <=79 : 
                         état_conducteur = "Moins bonne"
                    elif taux_cardiaque <80 : 
                        état_conducteur = "Mauvaise"

            # Détection de clignements et fatigue liées aux yeux
            current_time = time.time()
            if ear_général < SEUIL_ASPECT_YEUX:
                if not counter_condition:
                    counter += 1
                    message = "paupieres lourdes"
                    position = (10, 20)
                    font_famille = cv2.FONT_HERSHEY_TRIPLEX 
                    font_échelle = 0.8
                    coleur_msg = (0, 255, 255)
                    font_épaisseur = 1
                    cv2.putText(trame, message, position, font_famille, font_échelle, coleur_msg, font_épaisseur )

                    if counter > NB_TRAMES_SEUIL_ASPECT_YEUX:
                        cv2.putText(trame, "FATIGUE DETECTEE", (10, 50), cv2.FONT_HERSHEY_TRIPLEX,
                                   0.8, (0, 0, 255), 1)
            else:
                counter = 0
                counter_condition = False
                cv2.putText(trame, "Yeux Ouverts", (10, 30), cv2.FONT_HERSHEY_TRIPLEX, 0.8, (0, 255, 0), 1)
            cv2.putText(trame, f"cond_con : {counter_condition}", (300, 120), cv2.FONT_HERSHEY_TRIPLEX, 0.8, (0, 255, 255), 1)

            # Détecter un clignement 
            if ear_général < SEUIL_ASPECT_YEUX:
                if not etat_clignement:
                    clignement += 1
                    cv2.putText(trame, "clignement", (100, 30), cv2.FONT_HERSHEY_TRIPLEX, 0.8, (0, 255, 0), 1)
                    etat_clignement = True
            else:
                etat_clignement = False
            # Compter le nombre de clignement dans une minute à partir du 1er clignement
            if clignement == 1 and not etat_clignement:
                if not trigger_time_set_clignement:
                    temps_clignement = time.time()
                    trigger_time_set_clignement = True
            else:
                trigger_time_set_clignement = False
            # La condition pour détecter la fatigue à partir du nombre de clignement
            if temps_clignement != 0:
                if current_time - temps_clignement <= 60 and clignement > 20:
                    cv2.putText(trame, "FATIGUE", (10, 150), cv2.FONT_HERSHEY_TRIPLEX, 0.8, (0, 0, 255), 1)
                elif current_time - temps_clignement >= 60:
                    if clignement < 15:
                        cv2.putText(trame, "FATIGUE", (10, 150), cv2.FONT_HERSHEY_TRIPLEX, 0.8, (0, 255, 0), 1)
                    else:
                        clignement = 0
                        temps_clignement = 0
            
            cv2.putText(trame, f"temps_cli {current_time - temps_clignement}", (10, 200), cv2.FONT_HERSHEY_TRIPLEX, 0.8, (0, 0, 255), 1)
            # Afficher le nb de baillements actuel
            cv2.putText(trame, f"Nb_clignement : {clignement}", (300, 40), cv2.FONT_HERSHEY_TRIPLEX, 0.8, (0, 255, 255), 1)
            # Afficher la valeur du EAR actuelle
            cv2.putText(trame, f"EAR: {ear_formatté}", (10, 70), cv2.FONT_HERSHEY_TRIPLEX, 0.8, (0, 255, 0), 1)
            # Afficher la valeur du MAR actuelle
            cv2.putText(trame, f"MAR: {mar_formatté}", (10, 90), cv2.FONT_HERSHEY_TRIPLEX, 0.8, (0, 255, 255), 1)

            # Détection de baillements et fatigue liées à la bouche
            if mar_bouche > SEUIL_ASPECT_BOUCHE:
                cv2.putText(trame, "Baillement en cours", (300, 20), cv2.FONT_HERSHEY_TRIPLEX, 0.8, (0, 255, 255), 1)
                etat_clignement = True
                counter_condition = True
                if not état_baillement:
                    if not trigger_time_set:
                        yawning_time = current_time
                        trigger_time_set = True
                        temps_baillement = current_time  
                    elif current_time - yawning_time >= 3:
                        nb_baillements += 1
                        état_baillement = True
            else:
                état_baillement = False
                trigger_time_set = False
                yawning_time = 0  
           
            # Vérifier si le nombre de baillements a dépasser 2 pendant 7 minutes
            if time.time() - temps_baillement <= 4200 and nb_baillements > 2:
                cv2.putText(trame, "Fatigue detectee!", (300, 80), cv2.FONT_HERSHEY_TRIPLEX, 0.8, (0, 0, 255), 1)
            if time.time() - temps_baillement > 4200 and nb_baillements <= 2:
                nb_baillements = 0
            cv2.putText(trame, f"nb_baill : {nb_baillements}", (300, 60), cv2.FONT_HERSHEY_TRIPLEX, 0.8, (0, 255, 255), 1)
            
    cv2.imshow("Frame", trame)
            
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
        
cv2.destroyAllWindows()
caméra.release()